In [2]:
import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
import jax.numpy as jnp
import jax.random as random

# Set random seed
np.random.seed(42)
numpyro.set_platform("cpu")

# Generate toy data
# True latent dimension = 2, observed dimension = 5
n_samples = 100
n_features = 5
n_components = 2

# Create data with 2 latent factors
Z_true = np.random.randn(n_samples, n_components)
W_true = np.random.randn(n_components, n_features)
noise = 0.1 * np.random.randn(n_samples, n_features)
X = Z_true @ W_true + noise

# Probabilistic PCA model
def ppca_model(X, n_components):
    n_samples, n_features = X.shape
    
    # Priors
    W = numpyro.sample("W", dist.Normal(0, 1).expand([n_components, n_features]).to_event(2))
    sigma = numpyro.sample("sigma", dist.HalfNormal(1))
    
    # Latent variables
    with numpyro.plate("samples", n_samples):
        Z = numpyro.sample("Z", dist.Normal(0, 1).expand([n_components]).to_event(1))
        
        # Likelihood
        mu = Z @ W  # Shape: [n_samples, n_features]
        numpyro.sample("obs", dist.Normal(mu, sigma).to_event(1), obs=X)

# Run inference
rng_key = random.PRNGKey(0)
kernel = NUTS(ppca_model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=1000)
mcmc.run(rng_key, X, n_components)

# Get posterior samples
samples = mcmc.get_samples()

# Extract posterior means
W_posterior = samples["W"].mean(axis=0)
Z_posterior = samples["Z"].mean(axis=0)
sigma_posterior = samples["sigma"].mean()

# Reconstruct data
X_reconstructed = Z_posterior @ W_posterior

# Calculate reconstruction error
reconstruction_error = np.mean((X - X_reconstructed)**2)

# Print results
print("Original data shape:", X.shape)
print("Latent space shape:", Z_posterior.shape)
print("Loading matrix shape:", W_posterior.shape)
print(f"Noise level (sigma): {sigma_posterior:.3f}")
print(f"Reconstruction MSE: {reconstruction_error:.4f}")

# Compare true vs estimated latent factors (up to rotation/scaling)
# Calculate correlation between true and estimated latent variables
print("\nChecking latent space recovery:")
for i in range(n_components):
    correlations = [np.corrcoef(Z_true[:, j], Z_posterior[:, i])[0, 1] 
                   for j in range(n_components)]
    print(f"Component {i} max correlation with true factors: {max(abs(c) for c in correlations):.3f}")

sample: 100%|██████████| 1500/1500 [00:02<00:00, 521.01it/s, 127 steps of size 2.83e-02. acc. prob=0.93] 


Original data shape: (100, 5)
Latent space shape: (100, 2)
Loading matrix shape: (2, 5)
Noise level (sigma): 0.100
Reconstruction MSE: 0.0804

Checking latent space recovery:
Component 0 max correlation with true factors: 0.890
Component 1 max correlation with true factors: 0.936
